# School Measles (MMR) Vaccination Rates

**Goal:** predict a school's **MMR vaccination rate** from school characteristics
and geography, and explore where vaccination coverage falls below the ~95% herd-
immunity threshold.

This dataset is deliberately messy (heavy missingness and `-1` "not reported"
sentinel values), so the emphasis here is on **careful data cleaning** followed by
a leakage-free regression workflow:

1. Loading and data-quality assessment
2. Cleaning: sentinels, missingness, and feature decisions
3. Exploratory data analysis (EDA)
4. Preprocessing pipeline (imputation + scaling + encoding)
5. Model comparison, tuning, and evaluation
6. Feature importance and summary

**Dataset:** ~46,000 U.S. schools, 14 columns. Source: school-level MMR rates
(WSJ / data.world).

## 1. Setup and data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
# Update the path if needed
df = pd.read_csv("all-measles-rates.csv")
print(df.shape)
df.head()

## 2. Data-quality assessment and cleaning

### 2.1 What's missing, and where

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes,
    "nulls": df.isnull().sum(),
    "null_pct": (df.isnull().mean() * 100).round(1),
    "unique": df.nunique(),
})
quality

### 2.2 Handle the `-1` "not reported" sentinel

In this dataset, unreported `mmr` and `overall` rates are encoded as **-1**, not
as missing. Left untouched they would corrupt every average and model. We convert
them to `NaN`, then drop rows with no usable MMR target.

In [ ]:
for col in ["mmr", "overall"]:
    df[col] = df[col].replace(-1, np.nan)

print("Rows before:", len(df))
df = df.dropna(subset=["mmr"]).copy()
print("Rows with a usable MMR rate:", len(df))

df["mmr"].describe()

### 2.3 Clean the `year` column

`year` is stored as academic-year strings (e.g. `"2017-18"`) with some missing.
We extract the starting calendar year as a numeric feature.

In [ ]:
df["year_start"] = (
    df["year"].astype(str).str.extract(r"(\d{4})").astype(float)
)
print(df["year_start"].value_counts(dropna=False).sort_index())

### 2.4 Decide which columns to model on

- **Target:** `mmr`.
- **Drop `overall`** — it is almost a duplicate of the target (overall vs MMR
  coverage move together); using it would be leakage.
- **Drop the exemption columns** (`xrel`, `xmed`, `xper`) — they are 70-86%
  missing *and* mechanistically determine the rate; we keep the model honest by
  predicting coverage from school/geography instead. (They are explored in EDA.)
- **Drop identifiers / high-cardinality text** (`index`, `name`, `city`,
  `county`, `district`) that act as row IDs rather than generalizable signal.
- **Keep** `state`, `type`, `enroll`, `year_start`.

In [ ]:
numeric_features = ["enroll", "year_start"]
categorical_features = ["state", "type"]

X = df[numeric_features + categorical_features]
y = df["mmr"]
print("Feature matrix:", X.shape)

## 3. Exploratory data analysis

### 3.1 Distribution of MMR rates and the herd-immunity gap

In [ ]:
HERD_IMMUNITY = 95.0

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(df["mmr"], bins=50, color="#4c72b0")
axes[0].axvline(HERD_IMMUNITY, color="red", ls="--", lw=2, label="95% herd immunity")
axes[0].set_xlabel("MMR vaccination rate (%)")
axes[0].set_ylabel("Number of schools")
axes[0].set_title("Distribution of MMR rates")
axes[0].legend()

below = (df["mmr"] < HERD_IMMUNITY).mean()
axes[1].pie([below, 1 - below], labels=["Below 95%", "At/above 95%"],
            autopct="%1.1f%%", colors=["#c44e52", "#55a868"], startangle=90)
axes[1].set_title("Schools below herd-immunity threshold")
plt.tight_layout()
plt.show()

### 3.2 Coverage by state

In [ ]:
state_mmr = df.groupby("state")["mmr"].median().sort_values()
plt.figure(figsize=(10, 9))
sns.barplot(x=state_mmr.values, y=state_mmr.index, color="#4c72b0")
plt.axvline(HERD_IMMUNITY, color="red", ls="--", lw=2, label="95%")
plt.xlabel("Median MMR rate (%)")
plt.title("Median MMR vaccination rate by state")
plt.legend()
plt.tight_layout()
plt.show()

### 3.3 Coverage by school type, and exemptions vs coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

if df["type"].notna().any():
    order = df.groupby("type")["mmr"].median().sort_values().index
    sns.boxplot(data=df, x="type", y="mmr", order=order, ax=axes[0], color="#4c72b0")
    axes[0].axhline(HERD_IMMUNITY, color="red", ls="--", lw=1.5)
    axes[0].set_title("MMR rate by school type")
    axes[0].tick_params(axis="x", rotation=20)

# Personal-exemption rate vs MMR (where both are reported)
sub = df.dropna(subset=["xper"])
axes[1].scatter(sub["xper"], sub["mmr"], s=6, alpha=0.3, color="#c44e52")
axes[1].set_xlabel("Personal exemption rate (%)")
axes[1].set_ylabel("MMR rate (%)")
axes[1].set_title("Exemptions vs coverage (reported subset)")

plt.tight_layout()
plt.show()

## 4. Preprocessing pipeline

Because of the missing values, the pipeline imputes before scaling/encoding.
Numeric features get median imputation + standardization; categoricals get
most-frequent imputation + one-hot encoding. Everything is fit inside the
`Pipeline` on the training fold only, so there is no leakage.

In [ ]:
numeric_tf = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_tf = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_tf, numeric_features),
    ("cat", categorical_tf, categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 5. Model comparison (5-fold cross-validation)

We benchmark a linear baseline against two tree ensembles using cross-validated
R². Predicting coverage from school/geography alone is a genuinely hard problem,
so modest R² is expected and honest.

In [ ]:
candidates = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, random_state=RANDOM_STATE),
}

rows = []
for name, est in candidates.items():
    pipe = Pipeline([("prep", preprocessor), ("model", est)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2", n_jobs=-1)
    rows.append({"model": name, "cv_r2": scores.mean(), "cv_std": scores.std()})
    print(f"{name:<20} R2 {scores.mean():.4f} +/- {scores.std():.4f}")

cv_results = pd.DataFrame(rows).sort_values("cv_r2", ascending=False).reset_index(drop=True)
cv_results

### 5.1 Tune the top model (Random Forest)

In [ ]:
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])

param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", 0.5, 1.0],
}

search = RandomizedSearchCV(
    rf_pipe, param_distributions=param_dist, n_iter=20, cv=5,
    scoring="r2", random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)
search.fit(X_train, y_train)

print("Best CV R2:", round(search.best_score_, 4))
for k, v in search.best_params_.items():
    print(f"  {k} = {v}")

## 6. Evaluation on the held-out test set

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print("Test R2  :", round(r2_score(y_test, y_pred), 4))
print("Test MAE :", round(mean_absolute_error(y_test, y_pred), 4))
print("Test RMSE:", round(root_mean_squared_error(y_test, y_pred), 4))

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, s=8, alpha=0.3, color="#4c72b0")
lims = [df["mmr"].min(), 100]
plt.plot(lims, lims, "r--", lw=2)
plt.xlabel("Actual MMR rate (%)")
plt.ylabel("Predicted MMR rate (%)")
plt.title("Predicted vs actual")
plt.tight_layout()
plt.show()

## 7. Feature importance

In [ ]:
feature_names = best_model.named_steps["prep"].get_feature_names_out()
importances = best_model.named_steps["model"].feature_importances_

importance_df = (pd.DataFrame({"feature": feature_names, "importance": importances})
                 .sort_values("importance", ascending=False).reset_index(drop=True))

plt.figure(figsize=(9, 7))
sns.barplot(data=importance_df.head(15), x="importance", y="feature", color="#4c72b0")
plt.title("Top 15 features - MMR rate (tuned RF)")
plt.tight_layout()
plt.show()

importance_df.head(15)

## 8. Summary

- The raw data needed real cleaning: `-1` sentinels in `mmr`/`overall` were
  converted to `NaN`, academic-year strings were parsed to a numeric year, and
  leakage-prone or mostly-missing columns were excluded with documented reasoning.
- A pipeline with imputation handled the remaining missingness without leakage.
- Predicting coverage from school type, state, enrollment, and year is inherently
  hard; tree ensembles beat the linear baseline, with **state** and **enrollment**
  the most informative features.
- EDA highlights the public-health story: a meaningful share of schools sit below
  the 95% herd-immunity threshold, with wide variation across states.

**Possible extensions:** reframe as classification (below vs above 95%), add
county-level socioeconomic data, or model the exemption rates directly on the
reported subset.